# [Project title]

**Team members:** [Names]  
**Repository:** [URL of your team's fork]

## Setup

Use relative paths and keep the notebook runnable from beginning to end.

In [11]:
from pathlib import Path

DATA_DIR = Path("data")
assert DATA_DIR.is_dir(), "The data directory is missing."

In [12]:
# Imports
import pandas as pd

## Load data

In [13]:
# Load data from DATA_DIR.

df_census = pd.read_csv(DATA_DIR / "county_census_complete.csv")
df_lc_inc_county = pd.read_csv(DATA_DIR / "lung_cancer_incidence_per_county.csv")
df_smoking_prev = pd.read_csv(DATA_DIR / "PLACES__Local_Data_for_Better_Health,_County_Data_2023_release_20261025.csv")



## Prepare data

In [14]:
# Clean data and create features.
# county_complete

df_census.head()

# Columns to keep
# fips, name, state, pop_2019, uninsured_2019, unemployment_rate_2019, median_age_2019, median_individual_income_2019, median_individual_income_2019
cols_to_keep = [
    "fips",
    "name",
    "state",    
    "pop_2019",
    "uninsured_2019",
    "unemployment_rate_2019",
    "median_age_2019",
    "median_individual_income_2019"
]

clean_census = df_census[cols_to_keep]

# 3142 rows by 9 cols
print(clean_census.shape)

# Check to see any missing fips, county name, or state
for col in ["fips", "name", "state"]:
    n_nan = clean_census[col].isna().sum()
    n_empty = (clean_census[col].astype(str).str.strip() == "").sum()
    print(f"{col}: NaN={n_nan}, Empty={n_empty}")

# Expand to the rest of the dataset
for col in clean_census.columns:
    n_nan = clean_census[col].isna().sum()
    n_empty = (clean_census[col].astype(str).str.strip() == "").sum()
    print(f"{col}: NaN={n_nan}, Empty={n_empty}")

# Check for duplicates and length (should be 3143)
print("Rows:", len(clean_census)) # Missing one county
print("Unique FIPS:", clean_census["fips"].nunique())
print("Duplicate FIPS:", clean_census["fips"].duplicated().sum())




(3142, 8)
fips: NaN=0, Empty=0
name: NaN=0, Empty=0
state: NaN=0, Empty=0
fips: NaN=0, Empty=0
name: NaN=0, Empty=0
state: NaN=0, Empty=0
pop_2019: NaN=0, Empty=0
uninsured_2019: NaN=0, Empty=0
unemployment_rate_2019: NaN=0, Empty=0
median_age_2019: NaN=0, Empty=0
median_individual_income_2019: NaN=0, Empty=0
Rows: 3142
Unique FIPS: 3142
Duplicate FIPS: 0


In [15]:
# lung_cancer_incidence_per_county
df_lc_inc_county.head()

,Incidence Rate Report for United States by County,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Lung & Bronchus (All Stages^), 2019-2023",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"All Races (includes Hispanic), Both Sexes, All...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df_smoking_prev.head()
df_smoking_prev["Data_Value_Type"].unique()

is_smoking_measure = df_smoking_prev["MeasureId"] == "CSMOKING"
is_age_adjusted = df_smoking_prev["Data_Value_Type"] == "Age-adjusted prevalence"
county_smoking = df_smoking_prev[is_smoking_measure & is_age_adjusted].copy()

# keep and rename columns
county_smoking = county_smoking[["LocationID", "LocationName", "StateAbbr", "Year", "Data_Value"]]
county_smoking.columns = ["fips", "county_name", "state_abbr", "survey_year", "adult_smoking_pct"]
print("rows before EDA:", len(county_smoking))

# Check every column for NaN and empty strings before changing anything
for column in county_smoking.columns:
    n_nan = county_smoking[column].isna().sum()
    n_empty = (county_smoking[column].astype(str).str.strip() == "").sum()
    print(column, "| NaN:", n_nan, "| empty strings:", n_empty)

# turn empty strings into NaN, make smoking numeric, then drop rows missing anything
# (removes the national US row, which has no county name)
county_smoking = county_smoking.replace(r"^\s*$", pd.NA, regex=True)
county_smoking["adult_smoking_pct"] = pd.to_numeric(county_smoking["adult_smoking_pct"], errors="coerce")
rows_with_missing = county_smoking[county_smoking.isna().any(axis=1)]
print("Rows dropped for missing values:", len(rows_with_missing))
print(rows_with_missing)

county_smoking = county_smoking.dropna()
county_smoking = county_smoking[county_smoking["state_abbr"] != "US"]

# Clean FIPS (5-digit strings that represent US counties)
county_smoking["fips"] = county_smoking["fips"].astype(str).str.strip().str.zfill(5)

# Which states have no county data in the file
all_states = ["AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY"]
states_in_file = county_smoking["state_abbr"].unique()
states_missing = [state for state in all_states if state not in states_in_file]
print("States missing from the smoking data:", states_missing)
#florida is missing data


# final checks
print("Rows after cleaning:", len(county_smoking))
print("Missing values left:", county_smoking.isna().sum().sum())
print("Years:", county_smoking["survey_year"].unique())
print("FIPS lengths:", county_smoking["fips"].str.len().unique())
print("Duplicate counties:", county_smoking["fips"].duplicated().sum())

# Save
county_smoking.to_csv(DATA_DIR / "county_smoking_clean.csv", index=False)
county_smoking.head()



rows before EDA: 3077
fips | NaN: 0 | empty strings: 0
county_name | NaN: 1 | empty strings: 0
state_abbr | NaN: 0 | empty strings: 0
survey_year | NaN: 0 | empty strings: 0
adult_smoking_pct | NaN: 0 | empty strings: 0
Rows dropped for missing values: 1
       fips county_name state_abbr  survey_year  adult_smoking_pct
50871    59         NaN         US         2021               13.8
States missing from the smoking data: ['FL']
Rows after cleaning: 3076
Missing values left: 0
Years: [2021]
FIPS lengths: [5]
Duplicate counties: 0


,fips,county_name,state_abbr,survey_year,adult_smoking_pct
321,13095,Dougherty,GA,2021,21.0
596,28099,Neshoba,MS,2021,25.7
598,21097,Harrison,KY,2021,22.5
602,02066,Copper River,AK,2021,19.4
615,17151,Pope,IL,2021,21.0


## Explore data

In [17]:
# Summarize and visualize the prepared data.

## Build models

In [18]:
# Train baseline and candidate models.

## Evaluate models

In [19]:
# Compare models using the chosen evaluation measures.

## Final results

In [20]:
# Generate the final tables and figures.

## AI usage disclosure

- **AI tool/model:** [Name the tool and model, or state that no generative AI was used.]
- **Used for:** [Briefly identify the tasks.]
- **How we verified the work:** [Briefly identify the checks.]
- **Prompt(s) used:** [Paste the prompts used.]